<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/sessions/Session15_dialogue_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# جلسهٔ ۱۵: چت‌بات و سیستم دیالوگ وظیفه‌محور

### اهداف این جلسه

- تفاوت چت‌بات دامنه‌باز و سیستم دیالوگ وظیفه‌محور (TOD)
- مدیریت حافظهٔ گفتگو: پنجرهٔ لغزان و خلاصه‌سازی
- ساخت سیستم سفارش‌گیر با درک زبان، ردیابی وضعیت، سیاست و تولید پاسخ
- حل ابهام موجودیت با پرسش تکمیلی (Clarification) و اعتبارسنجی اسلات‌ها
- ارزیابی خودکار سیستم دیالوگ با شبیه‌ساز کاربر مبتنی بر LLM

> این نوت‌بوک به GPU نیاز دارد. در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۱. دو نوع سیستم دیالوگ

| | چت‌بات دامنه‌باز | سیستم دیالوگ وظیفه‌محور (TOD) |
|---|---|---|
| هدف | گفتگوی آزاد، پاسخ به هر پرسش | انجام یک کار مشخص: سفارش، رزرو، پیگیری |
| معیار موفقیت | رضایت کاربر | **انجام درست کار** در کمترین نوبت |
| خطای خطرناک | پاسخ نادرست | سفارش اشتباه، پول اشتباه |
| ساختار | یک مدل با تاریخچه | اجزای جدا با وضعیت صریح |

سیستم‌های TOD کلاسیک چهار جزء دارند:

1. **درک زبان (NLU):** قصد (Intent) و اسلات‌ها (Slots) را از پیام کاربر استخراج می‌کند.
2. **ردیابی وضعیت دیالوگ (DST):** وضعیت کل گفتگو را به‌روز نگه می‌دارد، مثل کالا، تعداد، رنگ و شهر.
3. **سیاست (Policy):** بر اساس وضعیت، اقدام بعدی سیستم را انتخاب می‌کند: پرسیدن، روشن‌سازی، تأیید یا ثبت.
4. **تولید زبان (NLG):** اقدام را به جملهٔ طبیعی تبدیل می‌کند.

با LLMها می‌توان همهٔ این‌ها را به یک مدل سپرد، ولی برای کاری که پول جابه‌جا می‌کند، **معماری ترکیبی** قابل‌اعتمادتر است: مدل زبان را می‌فهمد (NLU و DST)، و **کد** تصمیم می‌گیرد و عددها را می‌گوید (Policy و NLG قالب‌محور). این همان ایدهٔ جلسه‌های ۱۲ و ۱۴ است: کنترل در دست کد.

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/store_tools.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۲. حافظهٔ چت‌بات

در جلسهٔ ۱ دیدیم مدل حافظه ندارد و کل تاریخچه در هر درخواست ارسال می‌شود. در گفتگوی طولانی، تاریخچه گران می‌شود و از پنجرهٔ متن بیرون می‌زند. دو راه رایج:

- **پنجرهٔ لغزان:** فقط n پیام آخر. ساده است ولی اطلاعات اول گفتگو، مثل نام و آدرس، گم می‌شود.
- **خلاصه‌سازی:** پیام‌های قدیمی با یک فراخوانی مدل خلاصه می‌شوند و خلاصه جای آن‌ها را می‌گیرد.

</div>

In [ ]:
class SummaryMemory:
    """Keep the last `keep` messages verbatim and a running summary of everything older."""

    def __init__(self, system, keep=4):
        self.system, self.keep, self.summary, self.messages = system, keep, "", []

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})
        if len(self.messages) > self.keep:
            old, self.messages = self.messages[:-self.keep], self.messages[-self.keep:]
            transcript = "\n".join(f"{m['role']}: {m['content']}" for m in old)
            self.summary = chat(client, MODEL,
                                f"Previous summary: {self.summary or '(none)'}\n\nNew messages:\n{transcript}",
                                system=("Update the summary of this customer conversation in Persian. Keep every "
                                        "fact about the customer (name, city, products, preferences). Max 80 words."),
                                temperature=0)

    def context(self):
        system = self.system + (f"\n\nSummary of the earlier conversation: {self.summary}" if self.summary else "")
        return [{"role": "system", "content": system}] + self.messages


memory = SummaryMemory("تو دستیار فروشگاه نوین‌کالا هستی و کوتاه پاسخ می‌دهی.")
for user in ["سلام، من کیوان هستم از آمل.", "دنبال یه گوشی زیر ۳۰ میلیون هستم.",
             "دوربین برام مهمه.", "رنگ آبی رو بیشتر دوست دارم.", "خب، اسم من چی بود و کجا زندگی می‌کنم؟"]:
    memory.add("user", user)
    answer = chat(client, MODEL, messages=memory.context(), temperature=0)
    memory.add("assistant", answer)
    print(f"U: {user}\nA: {answer}\n")
print("SUMMARY:", memory.summary)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با خلاصه‌سازی، نام و شهر بعد از خارج شدن از پنجره هم حفظ می‌شوند. هزینه‌اش یک فراخوانی اضافه است و خلاصه ممکن است جزئیاتی را جا بیندازد. در سیستم‌های TOD راه بهتری داریم: اطلاعات مهم در **وضعیت صریح** نگه داشته می‌شوند، نه در متن.

## ۳. سیستم سفارش‌گیر نوین‌کالا

اجزای سیستم در یک سلول. هر جزء کوچک و قابل تست است:

- `understand` (NLU): مدل با خروجی ساختاریافته می‌گوید پیام جدید چه اطلاعاتی دارد. برای اینکه پاسخ‌های کوتاه مثل «آبی» یا «دومی» درست فهمیده شوند، وضعیت فعلی و آخرین پرسش سیستم هم به مدل داده می‌شود.
- `track` (DST): اطلاعات جدید را با وضعیت ادغام می‌کند. اگر کاربر کالای دیگری نام ببرد، انتخاب قبلی پاک می‌شود.
- `policy`: قواعد صریح. ابتدا کالا را پیدا می‌کند، بعد رنگ را با رنگ‌های موجود و تعداد را با موجودی اعتبارسنجی می‌کند، بعد تأیید می‌گیرد و در پایان سفارش را ثبت می‌کند.
- `respond` (NLG): جمله‌ها از قالب ساخته می‌شوند تا قیمت و نام کالا همیشه دقیقاً از دیتابیس بیایند.

</div>

In [ ]:
import json
from dataclasses import asdict, dataclass, field
from typing import Literal, Optional

from pydantic import BaseModel
from store_tools import get_product, place_order, search_products


class Update(BaseModel):
    """What the user's latest message says, in the context of the current state."""
    intent: Literal["order", "confirm", "deny", "other"]
    product_query: Optional[str]
    choice: Optional[int]
    quantity: Optional[int]
    color: Optional[str]
    city: Optional[str]


@dataclass
class State:
    product_query: Optional[str] = None
    product_id: Optional[str] = None
    candidates: list = field(default_factory=list)
    quantity: Optional[int] = None
    color: Optional[str] = None
    city: Optional[str] = None
    awaiting: Optional[str] = None          # "product", "choice", "color", "quantity", "city", "confirm"
    order: Optional[dict] = None


NLU_SYSTEM = """You update the state of an order dialogue for a Persian online store.
Given the current state, the assistant's last question and the user's new message, return what the NEW
message says. Leave a field null if the message does not mention it.
- intent: "order" when the user wants to buy or gives order details, "confirm" for yes/correct/ok
  (بله، آره، درسته، تأیید), "deny" for no/cancel (نه، خیر، لغو), otherwise "other".
- product_query: product words as written by the user (e.g. "گلکسی A55"), or null.
- choice: if the assistant listed numbered options and the user picks one ("دومی", "۲", "همون اولی"), its number.
- quantity: an integer if the user gives a number of items ("دو تا" -> 2, "یه" -> 1).
- color and city: in Persian."""


def understand(state, last_question, message):
    prompt = (f"STATE: {json.dumps(asdict(state), ensure_ascii=False)}\n"
              f"ASSISTANT ASKED: {last_question}\nUSER: {message}")
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": NLU_SYSTEM}, {"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "Update", "schema": Update.model_json_schema()}})
    return Update.model_validate_json(response.choices[0].message.content)


def track(state, update):
    """Dialogue state tracking: merge the update into the state."""
    if update.product_query and update.product_query != state.product_query:
        state.product_query, state.product_id, state.candidates = update.product_query, None, []
    if update.choice and state.candidates and 1 <= update.choice <= len(state.candidates):
        state.product_id = state.candidates[update.choice - 1]["product_id"]
    for slot in ("quantity", "color", "city"):
        if getattr(update, slot):
            setattr(state, slot, getattr(update, slot))
    return state


def policy(state, update):
    """Decide the next system action with explicit, testable rules."""
    if state.awaiting == "confirm":
        if update.intent == "confirm":
            try:
                state.order = place_order(state.product_id, state.quantity, state.city, state.color)
                return "placed", state.order
            except ValueError as e:
                return "order_failed", str(e)
        if update.intent == "deny":
            return "ask_change", None
    # 1. resolve the product (entity resolution with clarification)
    if state.product_id is None:
        if not state.product_query:
            return "ask_product", None
        state.candidates = search_products(state.product_query)
        in_stock = [c for c in state.candidates if c["stock"] > 0]
        if not in_stock:
            return "no_product", state.product_query
        if len(in_stock) == 1:
            state.product_id = in_stock[0]["product_id"]
        else:
            state.candidates = in_stock[:3]
            return "clarify_product", state.candidates
    product = get_product(state.product_id)
    # 2. validate and fill the remaining slots (colours compared without spaces / ZWNJ)
    same = lambda a, b: (a or "").replace("\u200c", "").replace(" ", "") == b.replace("\u200c", "").replace(" ", "")
    match = [c for c in product["colors"] if same(state.color, c)]
    if product["colors"] and not match:
        return "ask_color", (product, state.color)
    if match:
        state.color = match[0]                       # the catalogue spelling
    if not state.quantity:
        return "ask_quantity", None
    if state.quantity > product["stock"]:
        return "not_enough_stock", product
    if not state.city:
        return "ask_city", None
    return "confirm", product


def respond(action, data, state):
    """Template-based language generation: predictable, correct numbers."""
    if action == "ask_product":
        state.awaiting = "product"
        return "چه کالایی می‌خواهید سفارش بدهید؟"
    if action == "no_product":
        state.awaiting, state.product_query = "product", None
        return f"متأسفانه کالای موجودی برای «{data}» پیدا نکردم. لطفاً نام کالا را دقیق‌تر بنویسید."
    if action == "clarify_product":
        state.awaiting = "choice"
        options = "\n".join(f"{i}. {c['name']} ({c['price_toman']:,} تومان)" for i, c in enumerate(data, 1))
        return f"کدام کالا را می‌خواهید؟\n{options}"
    if action == "ask_color":
        product, wrong = data
        state.awaiting, state.color = "color", None
        prefix = f"رنگ «{wrong}» برای این کالا موجود نیست. " if wrong else ""
        return f"{prefix}{product['name']} در این رنگ‌ها موجود است: {'، '.join(product['colors'])}. کدام را می‌خواهید؟"
    if action == "ask_quantity":
        state.awaiting = "quantity"
        return "چند عدد می‌خواهید؟"
    if action == "not_enough_stock":
        state.awaiting, state.quantity = "quantity", None
        return f"از این کالا فقط {data['stock']} عدد موجود است. چند عدد می‌خواهید؟"
    if action == "ask_city":
        state.awaiting = "city"
        return "سفارش به کدام شهر ارسال شود؟"
    if action == "confirm":
        state.awaiting = "confirm"
        color = f"، رنگ {state.color}" if state.color else ""
        total = data["price_toman"] * state.quantity
        return (f"سفارش شما: {state.quantity} عدد {data['name']}{color}، ارسال به {state.city}. "
                f"مبلغ کالاها: {total:,} تومان. تأیید می‌کنید؟")
    if action == "ask_change":
        state.awaiting = None
        return "کدام مورد را می‌خواهید تغییر دهید؟"
    if action == "placed":
        state.awaiting = None
        return f"سفارش شما ثبت شد. کد پیگیری: {data['order_id']}. مبلغ: {data['total']:,} تومان."
    if action == "order_failed":
        state.awaiting = None
        return f"ثبت سفارش ممکن نشد: {data}"
    return "متوجه نشدم. لطفاً دوباره بگویید."


class OrderBot:
    def __init__(self):
        self.state, self.last = State(), "سلام! چه کالایی می‌خواهید سفارش بدهید؟"

    def turn(self, message):
        update = understand(self.state, self.last, message)
        track(self.state, update)
        action, data = policy(self.state, update)
        self.last = respond(action, data, self.state)
        return self.last, action

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### پرسش تکمیلی برای ابهام موجودیت

وقتی کاربر می‌گوید «یه گوشی سامسونگ می‌خوام»، چند کالا با این توصیف جور است. سیستم سه راه دارد: اولین نتیجه را حدس بزند، که خطرناک است، بگوید «پیدا نشد»، که ناامیدکننده است، یا **بپرسد** و گزینه‌ها را نشان دهد. پرسش تکمیلی (Clarification Question) یکی از موضوعات فعال پژوهش در سیستم‌های دیالوگ است. پرسش خوب کوتاه است، گزینه‌های واقعی را نشان می‌دهد و کاربر می‌تواند با یک کلمه جواب دهد.

</div>

In [ ]:
bot = OrderBot()
print("BOT:", bot.last)
for user in ["سلام، یه گوشی سامسونگ می‌خوام", "دومی", "قرمز داره؟", "آبی", "دو تا", "بفرستید شیراز", "بله درسته"]:
    reply, action = bot.turn(user)
    print(f"\nUSER: {user}\nBOT [{action}]: {reply}")
print("\nfinal state:", asdict(bot.state))

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

گفتگو را دنبال کنید. سیستم ابهام «گوشی سامسونگ» را با پرسش حل کرد، رنگ ناموجود را رد کرد و گزینه‌های درست را گفت، و پیش از ثبت، خلاصهٔ کامل سفارش را برای تأیید نشان داد. هر تصمیم در `policy` با یک قاعدهٔ صریح گرفته شده و قابل تست است. این همان چیزی است که در یک سیستم مالی لازم داریم.

کاربر می‌تواند همهٔ اطلاعات را در یک پیام هم بدهد. سیستم باید مستقیم به مرحلهٔ تأیید برود:

</div>

In [ ]:
bot = OrderBot()
for user in ["دو تا هدفون سونی XM5 مشکی برای تهران می‌خوام", "آره"]:
    reply, action = bot.turn(user)
    print(f"USER: {user}\nBOT [{action}]: {reply}\n")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## ۴. ارزیابی با شبیه‌ساز کاربر

ارزیابی سیستم دیالوگ سخت است: پاسخ هر نوبت به نوبت‌های قبل بستگی دارد و نمی‌توان آن را با یک مجموعهٔ ثابت پرسش و پاسخ سنجید. راه رایج، **شبیه‌ساز کاربر (User Simulator)** است: یک مدل زبانی نقش مشتری را بازی می‌کند که **هدف** مشخصی دارد. گفتگو تا ثبت سفارش یا سقف نوبت ادامه پیدا می‌کند و در پایان بررسی می‌کنیم سفارش ثبت‌شده با هدف یکی است یا نه.

معیارها:

- **نرخ موفقیت کار (Task Success):** سفارش درست ثبت شد؟
- **تعداد نوبت:** هرچه کمتر، بهتر، به شرط موفقیت.

</div>

In [ ]:
SIMULATOR = """You are a customer of an Iranian online store, chatting in informal Persian.
Your goal: buy {quantity} x {product}, colour {color}, delivered to {city}.
Rules: write short messages like a real customer. Do NOT give all details at once; start with only the
product. Answer the assistant's questions truthfully according to your goal. If the assistant offers
numbered options, pick the one that matches your goal. When the assistant summarises the order and it
matches your goal, confirm with «بله». If it does not match, say what is wrong."""

GOALS = [
    {"product": "گلکسی A55", "product_id": "P1001", "quantity": 1, "color": "آبی", "city": "آمل"},
    {"product": "هدفون سونی", "product_id": "P5001", "quantity": 2, "color": "نقره‌ای", "city": "تهران"},
    {"product": "پاوربانک شیائومی", "product_id": "P5005", "quantity": 3, "color": "مشکی", "city": "رشت"},
    {"product": "ساعت هوشمند سامسونگ", "product_id": "P5003", "quantity": 1, "color": "سبز", "city": "یزد"},
]


def simulate(goal, max_turns=10, verbose=False):
    bot = OrderBot()
    history = [{"role": "system", "content": SIMULATOR.format(**goal)},
               {"role": "user", "content": bot.last}]          # roles are swapped for the simulator
    for turn in range(1, max_turns + 1):
        user_message = chat(client, MODEL, messages=history, temperature=0.7)
        reply, action = bot.turn(user_message)
        if verbose:
            print(f"  customer: {user_message}\n  bot: {reply}")
        history += [{"role": "assistant", "content": user_message}, {"role": "user", "content": reply}]
        if action == "placed":
            s = bot.state
            ok = (s.product_id == goal["product_id"] and s.quantity == goal["quantity"]
                  and (s.color or "").replace("\u200c", " ") == goal["color"].replace("\u200c", " ")
                  and s.city == goal["city"])
            return {"success": ok, "turns": turn}
    return {"success": False, "turns": max_turns}


print(simulate(GOALS[0], verbose=True))

In [ ]:
import pandas as pd

results = pd.DataFrame([{**g, **simulate(g)} for g in GOALS for _ in range(2)])
print(f"task success: {results.success.mean():.0%}   mean turns: {results.turns.mean():.1f}")
results[["product", "quantity", "color", "city", "success", "turns"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

شبیه‌ساز هم یک مدل است و خطا دارد: گاهی هدفش را فراموش می‌کند یا همه چیز را یک‌جا می‌گوید. پس شکست را باید **بررسی** کرد: آیا سیستم ما اشتباه کرد یا شبیه‌ساز؟ دمای ۰٫۷ برای شبیه‌ساز باعث می‌شود سبک‌های مختلف کاربر را ببینیم. هر هدف را چند بار اجرا کنید.

شبیه‌ساز جایگزین آزمون با کاربر واقعی نیست، ولی برای **آزمون رگرسیون** عالی است: هر تغییری در سیاست یا NLU را می‌توان در چند دقیقه روی ده‌ها گفتگو سنجید.

## ۵. جمع‌بندی

- چت‌بات دامنه‌باز برای گفتگوی آزاد است و سیستم TOD برای انجام کار مشخص با معیار موفقیت روشن.
- حافظهٔ چت‌بات: پنجرهٔ لغزان ساده ولی فراموشکار است و خلاصه‌سازی اطلاعات قدیمی را فشرده نگه می‌دارد. در TOD، وضعیت صریح جای هر دو را می‌گیرد.
- معماری ترکیبی: مدل زبان را می‌فهمد (NLU و DST) و کد تصمیم می‌گیرد و عددها را می‌گوید (Policy و NLG قالب‌محور).
- پرسش تکمیلی ابهام موجودیت را حل می‌کند و از ثبت کالای اشتباه جلوگیری می‌کند.
- شبیه‌ساز کاربر ارزیابی خودکار و آزمون رگرسیون سیستم دیالوگ را ممکن می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

## تمرین‌ها

هر تمرین را در سلول کد زیر آن حل کنید. پاسخ پیشنهادی در پوشهٔ `solutions` آمده است.

۱. قصد جدیدی به نام `track_order` اضافه کنید: اگر کاربر کد پیگیری داد، وضعیت سفارش با ابزار `track_order` گزارش شود. NLU، سیاست و قالب پاسخ را به‌روز کنید.
۲. کاربر وسط گفتگو نظرش را عوض می‌کند: «نه، سه تا می‌خوام» یا «اصلاً آیفون می‌خوام». بررسی کنید سیستم فعلی این تغییرها را درست ردیابی می‌کند یا نه و اشکال‌ها را رفع کنید.
۳. تابع `respond` را با نسخه‌ای جایگزین کنید که پاسخ را با مدل زبانی تولید کند، با اقدام و داده به‌عنوان ورودی. پاسخ‌ها طبیعی‌تر می‌شوند؟ آیا عددها و نام‌ها همیشه درست می‌مانند؟
۴. شبیه‌ساز را روی ۱۰ هدف، هر کدام ۳ بار، اجرا کنید و همهٔ شکست‌ها را دسته‌بندی کنید: خطای NLU، خطای سیاست یا خطای شبیه‌ساز. کدام جزء بیشترین بهبود را لازم دارد؟

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۱.** قصد جدیدی به نام `track_order` اضافه کنید: اگر کاربر کد پیگیری داد، وضعیت سفارش با ابزار `track_order` گزارش شود. NLU، سیاست و قالب پاسخ را به‌روز کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۲.** کاربر وسط گفتگو نظرش را عوض می‌کند: «نه، سه تا می‌خوام» یا «اصلاً آیفون می‌خوام». بررسی کنید سیستم فعلی این تغییرها را درست ردیابی می‌کند یا نه و اشکال‌ها را رفع کنید.

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۳.** تابع `respond` را با نسخه‌ای جایگزین کنید که پاسخ را با مدل زبانی تولید کند، با اقدام و داده به‌عنوان ورودی. پاسخ‌ها طبیعی‌تر می‌شوند؟ آیا عددها و نام‌ها همیشه درست می‌مانند؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

**تمرین ۴.** شبیه‌ساز را روی ۱۰ هدف، هر کدام ۳ بار، اجرا کنید و همهٔ شکست‌ها را دسته‌بندی کنید: خطای NLU، خطای سیاست یا خطای شبیه‌ساز. کدام جزء بیشترین بهبود را لازم دارد؟

</div>

In [ ]:
# پاسخ خود را اینجا بنویسید